In [1]:
import numpy as np
class body:
    def __init__(self,name,pos):
        self.name=name
        self.pos=pos


b1=body("Earth",np.array([1,2,3]))
b1.pos

array([1, 2, 3])

In [34]:
import numpy as np
from scipy.integrate import solve_ivp
#This class contains the initial position and velocity of the celestial body:

class body:
    def __init__(self,name,mass,pos,vel):
        self.name=name
        self.mass=mass
        self.pos=np.array(pos,dtype=float)
        self.vel=np.array(vel,dtype=float)
        


b1 = body("Sun", 1.0, np.array([0, 0, 0]), np.array([0, 0, 0]))

b2 = body("Mercury", 1.660e-7, np.array([0.387, 0, 0]), np.array([0, 0.0278, 0]))

b3 = body("Venus", 2.448e-6, np.array([0.723, 0, 0]), np.array([0, 0.0202, 0]))

b4 = body("Earth", 3.003e-6, np.array([1.000, 0, 0]), np.array([0, 0.0172, 0]))

b5 = body("Mars", 3.227e-7, np.array([1.524, 0, 0]), np.array([0, 0.0139, 0]))

b6 = body("Jupiter", 9.545e-4, np.array([5.203, 0, 0]), np.array([0, 0.00754, 0]))

b7 = body("Saturn", 2.858e-4, np.array([9.537, 0, 0]), np.array([0, 0.00557, 0]))

b8 = body("Uranus", 4.366e-5, np.array([19.191, 0, 0]), np.array([0, 0.00396, 0]))

b9 = body("Neptune", 5.151e-5, np.array([30.069, 0, 0]), np.array([0, 0.00313, 0]))


bodies=[b1,b2,b3,b4,b5,b6,b7,b8,b9]


N=len(bodies)

def acc(mass,pos):

    
    a=np.zeros((len(bodies),3))

    for i in range(len(bodies)):
        for j  in range(len(bodies)):

            
            if i!=j:
                G = 2.959122082855911e-4
                
                r=pos[i]-pos[j]

                ax=-((G*mass[j]*r[0])/(np.linalg.norm(r)**3))
                ay=-((G*mass[j]*r[1])/(np.linalg.norm(r)**3))
                az=-((G*mass[j]*r[2])/(np.linalg.norm(r)**3))

                a[i]+=np.array([ax,ay,az])

    return a
                

            
def eq_o_mot(t,state):
    N=len(bodies)
    pos=state[:3*N].reshape(N,3)
    vel=state[3*N:].reshape(N,3)

    m=np.array([])
    for i in range(N):
        m=np.append(m,bodies[i].mass)

    ac=acc(m,pos)

    return np.concatenate((vel.flatten(),ac.flatten()))


initial_pos=np.array([])
initial_vel=np.array([])

for i in range(N):
    initial_pos=np.append(initial_pos,bodies[i].pos)
    initial_vel=np.append(initial_vel,bodies[i].vel)

state0=np.concatenate((initial_pos,initial_vel))

t_span = (0, 365.25)

t_eval = np.linspace(0, 365.25, 1000)
    
sol=solve_ivp(eq_o_mot,t_span,state0,method="RK45",t_eval=t_eval)

    
    


  

        

In [35]:
sol

  message: Required step size is less than spacing between numbers.
  success: False
   status: -1
        t: [ 0.000e+00  3.656e-01 ...  2.742e+02  2.746e+02]
        y: [[ 0.000e+00  9.397e-10 ...  4.294e-04  4.305e-04]
            [ 0.000e+00  7.625e-13 ...  8.464e-05  8.489e-05]
            ...
            [ 3.130e-03  3.130e-03 ...  3.129e-03  3.129e-03]
            [ 0.000e+00  0.000e+00 ...  0.000e+00  0.000e+00]]
      sol: None
 t_events: None
 y_events: None
     nfev: 3362
     njev: 0
      nlu: 0

In [36]:
len(sol.y[0])

752

In [30]:
print("Success:", sol.success)
print("Message:", sol.message)
print("Number of time points:", len(sol.t))
print("Final time:", sol.t[-1])

Success: False
Message: Required step size is less than spacing between numbers.
Number of time points: 752
Final time: 274.57732732732734


In [37]:
# Get the positions at the last successful time
last_pos = sol.y[:3*N, -1].reshape(N, 3)

min_dist = float("inf")

for i in range(N):
    for j in range(i + 1, N):
        distance = np.linalg.norm(last_pos[i] - last_pos[j])

        if distance < min_dist:
            min_dist = distance
            closest_pair = (bodies[i].name, bodies[j].name)

print("Last successful time:", sol.t[-1], "days")
print("Closest bodies:", closest_pair)
print("Distance:", min_dist, "AU")

Last successful time: 274.57732732732734 days
Closest bodies: ('Sun', 'Mercury')
Distance: 0.0019008503530956408 AU


In [38]:
mercury_index = 1

mercury_pos = sol.y[3*mercury_index:3*mercury_index+3, :].T
sun_pos = sol.y[0:3, :].T

distance = np.linalg.norm(mercury_pos - sun_pos, axis=1)

print("Initial distance:", distance[0], "AU")
print("Minimum distance:", np.min(distance), "AU")
print("Distance at final time:", distance[-1], "AU")

Initial distance: 0.387 AU
Minimum distance: 0.0019008503530956408 AU
Distance at final time: 0.0019008503530956408 AU
